# Configuration de l'environnement

In [ ]:
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE.ML;
CREATE WAREHOUSE IF NOT EXISTS ML_WH
  WAREHOUSE_SIZE = 'MEDIUM'
  AUTO_SUSPEND = 300;

In [ ]:
USE DATABASE HOUSE_PRICE;
USE SCHEMA ML;
USE WAREHOUSE ML_WH;

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark.session import Session
import pandas as pd

In [ ]:
session = get_active_session()
session.sql_simplifier_enabled = True

session.query_tag = {"origin":"Sharlen_D", 
                     "name":"Ml_houseprice", 
                     "version":{"major":1, "minor":0,},
                     "attributes":{"is_quickstart":1}}

print('Connection Established with the following parameters:')
print('User      : {}'.format(session.get_current_user()))
print('Role      : {}'.format(session.get_current_role()))
print('Database  : {}'.format(session.get_current_database()))
print('Schema    : {}'.format(session.get_current_schema()))
print('Warehouse : {}'.format(session.get_current_warehouse()))

# Ingestion des données depuis S3

## Le File Format JSON

In [ ]:
CREATE OR REPLACE FILE FORMAT JSON_FORMAT
  TYPE = 'JSON'
  STRIP_OUTER_ARRAY = TRUE     
  NULL_IF = ('NULL', 'null', '')
  COMPRESSION = 'AUTO';       

## Stage S3

In [ ]:
CREATE OR REPLACE STAGE HOUSE_STAGE
  URL = 's3://logbrain-datalake/datasets/house_price/'
  FILE_FORMAT = JSON_FORMAT;

## Table de staging en VARIANT

In [ ]:
CREATE OR REPLACE TABLE HOUSE_PRICES_VARIANT (
    RAW_DATA VARIANT
);

## Charger le JSON dans la table VARIANT

In [ ]:
COPY INTO HOUSE_PRICES_VARIANT
FROM @HOUSE_STAGE
FILE_FORMAT = JSON_FORMAT
ON_ERROR = 'CONTINUE';

-- Vérification
SELECT * FROM HOUSE_PRICES_VARIANT LIMIT 5;

## Table finale structurée

In [ ]:
CREATE OR REPLACE TABLE HOUSE_PRICES (
    PRICE            FLOAT,
    AREA             FLOAT,
    BEDROOMS         INT,
    BATHROOMS        INT,
    STORIES          INT,
    MAINROAD         VARCHAR(5),
    GUESTROOM        VARCHAR(5),
    BASEMENT         VARCHAR(5),
    HOTWATERHEATING  VARCHAR(5),
    AIRCONDITIONING  VARCHAR(5),
    PARKING          INT,
    PREFAREA         VARCHAR(5),
    FURNISHINGSTATUS VARCHAR(20)
);

In [ ]:
INSERT INTO HOUSE_PRICES
SELECT
    RAW_DATA:"price"::FLOAT            AS PRICE,
    RAW_DATA:"area"::FLOAT             AS AREA,
    RAW_DATA:"bedrooms"::INT           AS BEDROOMS,
    RAW_DATA:"bathrooms"::INT          AS BATHROOMS,
    RAW_DATA:"stories"::INT            AS STORIES,
    RAW_DATA:"mainroad"::VARCHAR       AS MAINROAD,
    RAW_DATA:"guestroom"::VARCHAR      AS GUESTROOM,
    RAW_DATA:"basement"::VARCHAR       AS BASEMENT,
    RAW_DATA:"hotwaterheating"::VARCHAR AS HOTWATERHEATING,
    RAW_DATA:"airconditioning"::VARCHAR AS AIRCONDITIONING,
    RAW_DATA:"parking"::INT            AS PARKING,
    RAW_DATA:"prefarea"::VARCHAR       AS PREFAREA,
    RAW_DATA:"furnishingstatus"::VARCHAR AS FURNISHINGSTATUS
FROM HOUSE_PRICES_VARIANT;

-- Vérification
SELECT COUNT(*) FROM HOUSE_PRICES;
SELECT * FROM HOUSE_PRICES LIMIT 5;

# Exploration des données avec Snowpark

In [ ]:
df = session.table("HOUSE_PRICES")

print(f"Lignes : {df.count()}")
df.show(5)
df.describe().show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pdf = df.to_pandas()

plt.figure(figsize=(12, 8))
sns.heatmap(pdf.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

In [ ]:
pdf["PRICE"].hist(bins=40, figsize=(8,4))
plt.title("Distribution des prix")
plt.xlabel("Prix")
plt.show()

# Préparation des features

In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

session = get_active_session()

# ── 1. Chargement ─────────────────────────────────────────────────────────
pdf = session.table("HOUSE_PRICES").to_pandas()
print(f"Shape : {pdf.shape}")

# ── 2. Encodage des colonnes catégorielles ────────────────────────────────
binary_cols = ["MAINROAD","GUESTROOM","BASEMENT",
               "HOTWATERHEATING","AIRCONDITIONING","PREFAREA"]
for col in binary_cols:
    pdf[col] = pdf[col].astype(str).str.strip().str.lower().map({"yes":1, "no":0})

pdf["FURNISHINGSTATUS"] = (
    pdf["FURNISHINGSTATUS"]
    .astype(str).str.strip().str.lower()
    .map({"furnished":2, "semi-furnished":1, "unfurnished":0})
)

# ── 3. Vérification NaN ───────────────────────────────────────────────────
assert pdf.isnull().sum().sum() == 0
print("✅ Aucun NaN")

# ── 4. Séparation X / y — PRIX RÉELS ─────────────────────────────────────
X = pdf.drop(columns=["PRICE"]).reset_index(drop=True)
y = pdf["PRICE"].reset_index(drop=True)
FEATURE_NAMES = X.columns.tolist()

print(f"\ny min : {y.min():,} | y max : {y.max():,} | y mean : {y.mean():,.0f}")

# ── 5. Scaling X uniquement ───────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=FEATURE_NAMES)
print("✅ Scaling OK")

# ── 6. Split train/test ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42

)

print(f"\nTrain : {X_train.shape} | Test : {X_test.shape}")
print(f"y_train sample : {y_train.values[:5]}")
print(f"y_train min/max : {y_train.min():,} / {y_train.max():,}")

## Test Modèle 1 : XGBoost Regressor

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ── Entraînement ──────────────────────────────────────────────────────────
xgb_reg = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    verbosity=0
)
xgb_reg.fit(X_train, y_train)
print("✅ XGBoost Regressor entraîné")

# ── Prédictions ───────────────────────────────────────────────────────────
y_pred_xgb = xgb_reg.predict(X_test)

# ── Métriques ─────────────────────────────────────────────────────────────
mae  = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2   = r2_score(y_test, y_pred_xgb)

print("\n" + "="*40)
print("  MÉTRIQUES — XGBoost Regressor")
print("="*40)
print(f"  MAE  : {mae:>12,.0f}")
print(f"  RMSE : {rmse:>12,.0f}")
print(f"  R²   : {r2:>12.4f}")
print("="*40)

## Test Modèle 2 : Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ── Entraînement ──────────────────────────────────────────────────────────
lr_reg = LinearRegression()
lr_reg.fit(X_train, y_train)
print("✅ Linear Regression entraînée")

# ── Prédictions ───────────────────────────────────────────────────────────
y_pred_lr = lr_reg.predict(X_test)

# ── Métriques ─────────────────────────────────────────────────────────────
mae  = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2   = r2_score(y_test, y_pred_lr)

print("\n" + "="*40)
print("  MÉTRIQUES — Linear Regression")
print("="*40)
print(f"  MAE  : {mae:>12,.0f}")
print(f"  RMSE : {rmse:>12,.0f}")
print(f"  R²   : {r2:>12.4f}")
print("="*40)

# ── Comparaison des coefficients ──────────────────────────────────────────
import pandas as pd

coefs = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr_reg.coef_
}).sort_values("Coefficient", ascending=False)

print("\n── Importance des features (coefficients) ──")
print(coefs.to_string(index=False))

## Comparaison finale des modèles

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Données ───────────────────────────────────────────────────────────────
modeles = ["XGBoost Regressor", "Linear Regression"]

mae  = [22293,  40253]
rmse = [32694,  53985]
r2   = [0.8801, 0.6732]

# ── Tableau comparatif ────────────────────────────────────────────────────
print("="*50)
print(f"{'Métrique':<12} {'XGBoost':>16} {'Linear Reg':>16}")
print("="*50)
print(f"{'MAE':<12} {mae[0]:>16,.0f} {mae[1]:>16,.0f}")
print(f"{'RMSE':<12} {rmse[0]:>16,.0f} {rmse[1]:>16,.0f}")
print(f"{'R²':<12} {r2[0]:>16.4f} {r2[1]:>16.4f}")
print("="*50)


# ── Graphique ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
colors = ["steelblue", "coral"]

for ax, valeurs, titre, meilleur_bas in zip(
    axes,
    [mae, rmse, r2],
    ["MAE ↓", "RMSE ↓", "R² ↑"],
    [True, True, False]
):
    bars = ax.bar(modeles, valeurs, color=colors, width=0.5)
    ax.set_title(titre, fontsize=13, fontweight="bold")
    ax.bar_label(bars, fmt=lambda x: f"{x:,.0f}" if x > 1 else f"{x:.4f}", padding=5)
    ax.set_ylim(0, max(valeurs) * 1.2)
    # Surligner le gagnant
    idx_winner = int(not meilleur_bas) if meilleur_bas else int(valeurs[1] > valeurs[0])
    bars[idx_winner].set_edgecolor("gold")
    bars[idx_winner].set_linewidth(3)

plt.suptitle("Comparaison XGBoost vs Linear Regression", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Entraînement de plusieurs modèles

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

models = {
    "LinearRegression":     LinearRegression(),
    "RandomForest":         RandomForestRegressor(n_estimators=200, random_state=42),
    "GradientBoosting":     GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost":              XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        "MAE":  round(mean_absolute_error(y_test, preds), 2),
        "RMSE": round(np.sqrt(mean_squared_error(y_test, preds)), 2),
        "R2":   round(r2_score(y_test, preds), 4)
    }

# Affichage comparatif
import pandas as pd
pd.DataFrame(results).T.sort_values("R2", ascending=False)

## Cross-Validation (5-fold) sur XGBoost

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    XGBRegressor(n_estimators=200, random_state=42, verbosity=0),
    X_scaled, y,
    cv=5,
    scoring="r2"
)
print(f"R² moyen (5-fold CV) : {scores.mean():.4f} ± {scores.std():.4f}")

## Analyse des outliers

In [ ]:
# Visualiser les outliers
Q1, Q3 = y.quantile(0.25), y.quantile(0.75)
IQR = Q3 - Q1
outliers = y[(y < Q1 - 1.5*IQR) | (y > Q3 + 1.5*IQR)]
print(f"Outliers détectés : {len(outliers)} ({len(outliers)/len(y)*100:.1f}%)")

# Filtrer pour l'entraînement
mask = (y >= Q1 - 1.5*IQR) & (y <= Q3 + 1.5*IQR)
X_clean, y_clean = X_scaled[mask], y[mask]
print(f"Données après filtre : {X_clean.shape[0]} lignes")

## Optimisation des hyperparamètres — RandomForest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid_rf = {
    "n_estimators":      [100, 200, 300],
    "max_depth":         [3, 5, 7, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
}

rf = RandomForestRegressor(random_state=42)
search_rf = RandomizedSearchCV(
    rf, param_grid_rf,
    n_iter=20,
    cv=5,
    scoring="r2",
    n_jobs=1,
    random_state=42,
    verbose=1
)
search_rf.fit(X_train, y_train)

best_model = search_rf.best_estimator_
preds_best = best_model.predict(X_test)

print("Meilleurs paramètres :", search_rf.best_params_)
print(f"R² optimisé RF       : {r2_score(y_test, preds_best):.4f}")

# Stockage dans le Snowflake Model Registry

In [ ]:
import json
from snowflake.ml.registry import Registry
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datetime import datetime

DB         = "HOUSE_PRICE"
SCHEMA     = "ML"
MODEL_XGB  = "HOUSE_PRICE_XGBOOST"
MODEL_RF   = "HOUSE_PRICE_RF"
ALIAS_PROD = "prod"

version_name = f"V{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Version : {version_name}")

def rotate_alias(session, db, schema, model_name,
                  new_version, alias="prod", reg=None):
    fqn = f"{db}.{schema}.{model_name}"

    # Étape A — trouver et dropper l'alias existant via SQL
    try:
        rows = session.sql(
            f"SHOW VERSIONS IN MODEL {fqn}"
        ).collect()
        for row in rows:
            d       = row.as_dict()
            version = d.get("name", "")
            aliases = json.loads(d.get("aliases", "[]") or "[]")
            if alias.lower() in [a.lower() for a in aliases]:
                session.sql(
                    f'ALTER MODEL {fqn} '
                    f'MODIFY VERSION "{version}" DROP ALIAS {alias}'
                ).collect()
                print(f"  retiré de '{version}'")
    except Exception as e:
        print(f"  pas d'alias à retirer : {e}")

    # Étape B — poser l'alias via API Python Registry
    try:
        reg.get_model(model_name).version(new_version).set_alias(alias)
        print(f"  alias '{alias}' → '{new_version}'")
    except Exception as e:
        print(f"  échec set_alias : {e}")

def purge_old_versions(session, db, schema, model_name, keep=3):
    fqn  = f"{db}.{schema}.{model_name}"
    rows = session.sql(f"SHOW VERSIONS IN MODEL {fqn}").collect()
    versioned = sorted(
        [r.as_dict() for r in rows],
        key=lambda x: x.get("created_on", ""),
        reverse=True
    )
    for i, d in enumerate(versioned):
        v       = d.get("name", "")
        aliases = json.loads(d.get("aliases", "[]") or "[]")
        if i < keep or bool(aliases):
            continue
        try:
            session.sql(
                f'ALTER MODEL {fqn} DROP VERSION "{v}"'
            ).collect()
            print(f"  supprimée : {v}")
        except Exception as e:
            print(f"  ignorée : {v} → {e}")

reg      = Registry(session=session, database_name=DB, schema_name=SCHEMA)
X_sample = X_test.iloc[:5].reset_index(drop=True)

best_xgb  = models["XGBoost"]
preds_xgb = best_xgb.predict(X_test)
metrics_xgb = {
    "R2":   round(r2_score(y_test, preds_xgb), 4),
    "MAE":  round(mean_absolute_error(y_test, preds_xgb), 2),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, preds_xgb)), 2),
}

reg.log_model(
    model=best_xgb, model_name=MODEL_XGB,
    version_name=version_name,
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    sample_input_data=X_sample,
    comment=f"XGBoost R²={metrics_xgb['R2']}",
    metrics=metrics_xgb,
    options={"relax_version": False},
)
print(f"XGBoost enregistré : {version_name}")

rotate_alias(session, DB, SCHEMA, MODEL_XGB,
             version_name, alias=ALIAS_PROD, reg=reg)

purge_old_versions(session, DB, SCHEMA, MODEL_XGB, keep=3)

preds_best = best_model.predict(X_test)
metrics_rf = {
    "R2":   round(r2_score(y_test, preds_best), 4),
    "MAE":  round(mean_absolute_error(y_test, preds_best), 2),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, preds_best)), 2),
}

reg.log_model(
    model=best_model, model_name=MODEL_RF,
    version_name=version_name,
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    sample_input_data=X_sample,
    comment=f"RandomForest R²={metrics_rf['R2']}",
    metrics=metrics_rf,
    options={"relax_version": False},
)
print(f"RandomForest enregistré : {version_name}")

purge_old_versions(session, DB, SCHEMA, MODEL_RF, keep=3)


# Vérification finale

print("\n=== Modèles dans le Registry ===")
for m in reg.models():
    print(f"  - {m.name}")

print("\n=== Versions XGBoost ===")
rows = session.sql(
    f"SHOW VERSIONS IN MODEL {DB}.{SCHEMA}.{MODEL_XGB}"
).collect()
for row in rows:
    d       = row.as_dict()
    aliases = json.loads(d.get("aliases", "[]") or "[]")
    print(f"  {d['name']:30s} → aliases : {aliases}")

# Inférence depuis le Registry

In [ ]:
# ── Charger le modèle en production (alias prod) ─────────────────────────
model_prod = reg.get_model("HOUSE_PRICE_XGBOOST").version("prod")
print(f"✅ Modèle chargé : {model_prod.model_name} ({model_prod.version_name})")

# ── Préparer 3 nouvelles maisons (non vues à l'entraînement) ─────────────
# IMPORTANT : utiliser exactement FEATURE_NAMES (les 15 colonnes avec features engineered)
new_houses = pd.DataFrame([
    {   # Maison 1 — grande, bien équipée, zone privilégiée
        "AREA": 250, "BEDROOMS": 4, "BATHROOMS": 3, "STORIES": 3,
        "MAINROAD": 1, "GUESTROOM": 1, "BASEMENT": 1,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 1,
        "PARKING": 2, "PREFAREA": 1, "FURNISHINGSTATUS": 2,
        "AREA_PER_BEDROOM": 250 / (4 + 1),
        "COMFORT_SCORE": 1 + 0 + 1 + 1,
        "AREA_X_STORIES": 250 * 3,
    },
    {   # Maison 2 — petite, non meublée
        "AREA": 80, "BEDROOMS": 2, "BATHROOMS": 1, "STORIES": 1,
        "MAINROAD": 0, "GUESTROOM": 0, "BASEMENT": 0,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 0,
        "PARKING": 0, "PREFAREA": 0, "FURNISHINGSTATUS": 0,
        "AREA_PER_BEDROOM": 80 / (2 + 1),
        "COMFORT_SCORE": 0,
        "AREA_X_STORIES": 80 * 1,
    },
    {   # Maison 3 — moyenne gamme
        "AREA": 150, "BEDROOMS": 3, "BATHROOMS": 2, "STORIES": 2,
        "MAINROAD": 1, "GUESTROOM": 0, "BASEMENT": 0,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 1,
        "PARKING": 1, "PREFAREA": 0, "FURNISHINGSTATUS": 1,
        "AREA_PER_BEDROOM": 150 / (3 + 1),
        "COMFORT_SCORE": 1 + 0 + 0 + 0,
        "AREA_X_STORIES": 150 * 2,
    },
], columns=FEATURE_NAMES)  # Respect strict de l'ordre des colonnes

# ── Appliquer le même scaler que l'entraînement ───────────────────────────
new_houses_scaled = pd.DataFrame(
    scaler.transform(new_houses),
    columns=FEATURE_NAMES
)

# ── Inférence ─────────────────────────────────────────────────────────────
predictions = model_prod.run(new_houses_scaled, function_name="predict")
preds_flat = predictions.values.flatten()

print("\n=== Résultats de l'inférence ===")
for i, pred in enumerate(preds_flat):
    print(f"  Maison {i+1} : {pred:,.0f} €")

# Ordre de grandeur de référence
print(f"\nRéférence y_test → min: {y_test.min():,} | max: {y_test.max():,} | mean: {y_test.mean():,.0f}")

# ── Stocker les prédictions dans une table Snowflake ──────────────────────
new_houses_scaled["PREDICTED_PRICE"] = preds_flat.astype(float)

results_df = session.create_dataframe(new_houses_scaled)
results_df.write.mode("overwrite").save_as_table("HOUSE_PRICE.ML.HOUSE_PRICES_PREDICTIONS")
print("✅ Prédictions stockées dans HOUSE_PRICES_PREDICTIONS")

# ── Vérification ──────────────────────────────────────────────────────────
session.table("HOUSE_PRICE.ML.HOUSE_PRICES_PREDICTIONS").show()

# Application Streamlit in Snowflake

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from sklearn.preprocessing import StandardScaler

st.title("🏠 Estimation du Prix d'une Maison")
st.markdown("Renseignez les caractéristiques de la maison pour obtenir une estimation de prix.")

session = get_active_session()

@st.cache_resource
def load_scaler_and_model(model_alias="prod"):

    pdf = session.table("HOUSE_PRICE.ML.HOUSE_PRICES").to_pandas()

    binary_cols = ["MAINROAD","GUESTROOM","BASEMENT",
                   "HOTWATERHEATING","AIRCONDITIONING","PREFAREA"]
    for col in binary_cols:
        pdf[col] = pdf[col].astype(str).str.strip().str.lower().map({"yes":1,"no":0})
    pdf["FURNISHINGSTATUS"] = pdf["FURNISHINGSTATUS"].astype(str).str.strip().str.lower().map({
        "furnished":2, "semi-furnished":1, "unfurnished":0
    })

    X = pdf.drop(columns=["PRICE"])
    feature_names = X.columns.tolist()

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    scaler.fit(X)

    reg = Registry(session=session,
                   database_name="HOUSE_PRICE",
                   schema_name="ML")
    model = reg.get_model("HOUSE_PRICE_XGBOOST").version("prod")

    return scaler, feature_names, model

MODEL_ALIAS = "prod"
scaler, FEATURE_NAMES, model = load_scaler_and_model(MODEL_ALIAS)


st.subheader("📋 Caractéristiques de la maison")

col1, col2, col3 = st.columns(3)

with col1:
    area     = st.number_input("Surface (m²)",    min_value=30,  max_value=400, value=150, step=5)
    bedrooms = st.slider("Chambres",              min_value=1,   max_value=6,   value=3)
    bathrooms= st.slider("Salles de bain",        min_value=1,   max_value=4,   value=2)
    stories  = st.slider("Étages",                min_value=1,   max_value=4,   value=2)

with col2:
    parking      = st.slider("Places parking",    min_value=0,   max_value=3,   value=1)
    mainroad     = st.selectbox("Route principale",   ["yes", "no"])
    guestroom    = st.selectbox("Chambre d'amis",     ["no", "yes"])
    basement     = st.selectbox("Sous-sol",           ["no", "yes"])

with col3:
    hotwaterheating = st.selectbox("Chauffe-eau",         ["no", "yes"])
    airconditioning = st.selectbox("Climatisation",       ["no", "yes"])
    prefarea        = st.selectbox("Zone privilégiée",    ["no", "yes"])
    furnishing      = st.selectbox("Ameublement", 
                                   ["furnished", "semi-furnished", "unfurnished"])


def yn(v): return 1 if v == "yes" else 0
furn_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}

if st.button("💡 Estimer le prix", use_container_width=True):
    input_df = pd.DataFrame([{
        "AREA":             area,
        "BEDROOMS":         bedrooms,
        "BATHROOMS":        bathrooms,
        "STORIES":          stories,
        "MAINROAD":         yn(mainroad),
        "GUESTROOM":        yn(guestroom),
        "BASEMENT":         yn(basement),
        "HOTWATERHEATING":  yn(hotwaterheating),
        "AIRCONDITIONING":  yn(airconditioning),
        "PARKING":          parking,
        "PREFAREA":         yn(prefarea),
        "FURNISHINGSTATUS": furn_map[furnishing]
    }], columns=FEATURE_NAMES)

    input_scaled = pd.DataFrame(
        scaler.transform(input_df),
        columns=FEATURE_NAMES
    )

    prediction = model.run(input_scaled, function_name="predict")
    price = float(prediction.values.flatten()[0])

    st.divider()
    st.success(f"### 💰 Prix estimé : {price:,.0f} €")


    min_p, max_p = 87500, 665000
    pct = min(max((price - min_p) / (max_p - min_p), 0), 1)
    st.progress(pct, text=f"Positionnement dans la fourchette du marché ({pct*100:.0f}%)")


    with st.expander("📊 Récapitulatif des caractéristiques saisies"):
        st.dataframe(input_df)

    st.balloons()


st.divider()
st.caption("Modèle : XGBoost | Données : 1090 maisons")

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION & SESSION ---
# Dans un Notebook Snowflake, la session est souvent déjà disponible via 'session'
session = get_active_session()

# --- CUSTOM CSS ---
# On conserve votre style premium pour garder l'aspect "App" dans le notebook
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&display=swap');

html, body, [class*="css"] { font-family: 'DM Sans', sans-serif; }
.stApp { background: #f5f3ef; }

.hero {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
    border-radius: 20px;
    padding: 30px;
    margin-bottom: 25px;
    color: white;
}
.hero-title { font-size: 2.2rem; margin: 0; color: #ffffff; }
.hero-title span { color: #e5b80b; }
.hero-badge {
    background: rgba(229,184,11,0.15);
    border: 1px solid rgba(229,184,11,0.4);
    color: #e5b80b;
    font-size: 0.7rem;
    padding: 2px 10px;
    border-radius: 10px;
    text-transform: uppercase;
}

.card {
    background: #ffffff;
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.05);
    margin-bottom: 15px;
    border: 1px solid #eee;
}
.result-card {
    background: linear-gradient(135deg, #1a1a2e 0%, #0f3460 100%);
    border-radius: 15px;
    padding: 25px;
    text-align: center;
    color: white;
}
.result-price { font-size: 3rem; color: #e5b80b; font-weight: bold; }
</style>
""", unsafe_allow_html=True)

# --- CHARGEMENT DES DONNÉES ET DU MODÈLE ---
@st.cache_resource
def load_assets():
    # Récupération des données pour fitter le scaler (comme dans votre original)
    df_train = session.table("HOUSE_PRICE.ML.HOUSE_PRICES").to_pandas()
    
    # Prétraitement identique pour le fit du scaler
    binary_cols = ["MAINROAD", "GUESTROOM", "BASEMENT", "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA"]
    for col in binary_cols:
        df_train[col] = df_train[col].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
    
    df_train["FURNISHINGSTATUS"] = (
        df_train["FURNISHINGSTATUS"].astype(str).str.strip().str.lower()
        .map({"furnished": 2, "semi-furnished": 1, "unfurnished": 0})
    )
    
    X = df_train.drop(columns=["PRICE"])
    feature_names = X.columns.tolist()
    
    scaler = StandardScaler()
    scaler.fit(X)
    
    # Accès au registre Snowflake ML
    reg = Registry(session=session, database_name="HOUSE_PRICE", schema_name="ML")
    model = reg.get_model("HOUSE_PRICE_XGBOOST").version("prod")
    
    return scaler, feature_names, model

try:
    scaler, FEATURE_NAMES, model = load_assets()
except Exception as e:
    st.error(f"Erreur de chargement : Vérifiez que la table et le modèle existent. {e}")
    st.stop()

# Helpers
YN = lambda v: 1 if v == "yes" else 0
FURN_MAP = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}
MIN_P, MAX_P = 87_500, 665_000

# --- INTERFACE ---
st.markdown("""
<div class="hero">
  <span class="hero-badge">Snowflake ML Registry</span>
  <h1 class="hero-title">Estimateur <span>Immobilier</span></h1>
</div>
""", unsafe_allow_html=True)

# Formulaire d'entrée
with st.container():
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.markdown('<div class="card"><b>Dimensions</b>', unsafe_allow_html=True)
        area = st.number_input("Surface (m²)", 30, 400, 150)
        bedrooms = st.slider("Chambres", 1, 6, 3)
        bathrooms = st.slider("Salles de bain", 1, 4, 2)
        st.markdown('</div>', unsafe_allow_html=True)

    with col2:
        st.markdown('<div class="card"><b>Équipements</b>', unsafe_allow_html=True)
        parking = st.slider("Parking", 0, 3, 1)
        airco = st.checkbox("Climatisation", value=True)
        guest = st.checkbox("Chambre d'amis")
        basement = st.checkbox("Sous-sol")
        st.markdown('</div>', unsafe_allow_html=True)

    with col3:
        st.markdown('<div class="card"><b>Standing</b>', unsafe_allow_html=True)
        furnishing = st.selectbox("Ameublement", ["furnished", "semi-furnished", "unfurnished"])
        prefarea = st.checkbox("Zone privilégiée")
        mainroad = st.checkbox("Route principale", value=True)
        hotwater = st.checkbox("Eau chaude")
        st.markdown('</div>', unsafe_allow_html=True)

# --- CALCUL ---
if st.button("🚀 Calculer l'estimation", use_container_width=True):
    # Mapping des booléens/checkbox pour le modèle
    input_data = pd.DataFrame([{
        "AREA": area,
        "BEDROOMS": bedrooms,
        "BATHROOMS": bathrooms,
        "STORIES": 2, # Valeur par défaut ou ajouter un slider
        "MAINROAD": 1 if mainroad else 0,
        "GUESTROOM": 1 if guest else 0,
        "BASEMENT": 1 if basement else 0,
        "HOTWATERHEATING": 1 if hotwater else 0,
        "AIRCONDITIONING": 1 if airco else 0,
        "PARKING": parking,
        "PREFAREA": 1 if prefarea else 0,
        "FURNISHINGSTATUS": FURN_MAP[furnishing],
    }], columns=FEATURE_NAMES)

    # Scaling
    input_scaled = pd.DataFrame(scaler.transform(input_data), columns=FEATURE_NAMES)

    # Prédiction via Snowflake ML
    with st.spinner("Interrogation du modèle..."):
        prediction = model.run(input_scaled, function_name="predict")
        price = float(prediction.values.flatten()[0])

    # Affichage du résultat
    st.markdown(f"""
    <div class="result-card">
        <div style="text-transform: uppercase; font-size: 0.8rem; opacity: 0.7;">Valeur Estimée</div>
        <div class="result-price">{price:,.0f} €</div>
    </div>
    """, unsafe_allow_html=True)
    
    st.balloons()

# Footer discret
st.caption("Notebook Snowflake - Modèle XGBoost hébergé sur Snowflake Registry")